In [1]:
import numpy as np
import torch
import time
import math

torch.set_printoptions(8)

In [2]:
GELU_PARA1 = math.sqrt(2.0 / math.pi)
GELU_PARA2 = 0.044715


def gelu(x):
    """
        Task: Use the torch API to implement the approximate calculation formula of the `GELU`
        activation function. The formula is as follows (you need to paste it into the latex
        online conversion website)
        Website: https://www.latexlive.com/
        Formula: \frac{1}{2} x\left[1+\tanh \left(\sqrt{\frac{2}{\pi}}\left(x+0.044715 x^{3}\right)\right)\right]

        Input: Tensor
        Output: Tensor
    """
    y = 0.5 * x * (1 + torch.tanh(GELU_PARA1 * (x + GELU_PARA2 * x ** 3)))
    return y

In [5]:
x_test = torch.randn(10)
my_result = gelu(x_test)
correct_result = torch.nn.functional.gelu(x_test)

diff = (my_result - correct_result).abs()
print(f"最大差异: {diff.max().item():.10f}")
print(f"平均差异: {diff.mean().item():.10f}")

最大差异: 0.0002333745
平均差异: 0.0000921063


In [6]:
def softmax(x, dim=-1):  #处理任意维张量，默认是最后一维
    """
        softmax公式：softmax(x_i) = exp(x_i - max(x)) / sum(exp(x_j - max(x)))
    """
    #keepdim=True：保留原来的维度结构
    #[0]表示要取最大值本身，而不是最大值的索引
    x_max = x.max(dim=dim, keepdim=True)[0]
    x_new = x - x_max
    x_exp = torch.exp(x_new)
    #再dim维上求和
    x_sum = x_exp.sum(dim=dim, keepdim=True)
    return x_exp / x_sum

In [7]:
# 测试1：一维
x1 = torch.tensor([1.0, 2.0, 3.0])
print(softmax(x1))  # 输出: tensor([0.0900, 0.2447, 0.6652])
print(torch.sum(softmax(x1)))  # 输出: tensor(1.0000)

# 测试2：二维（对最后一维做softmax）
x2 = torch.randn(2, 4)
my_result = softmax(x2, dim=-1)
official_result = torch.softmax(x2, dim=-1)
print(torch.allclose(my_result, official_result))  # 应该输出 True

# 测试3：比较官方实现
x3 = torch.randn(10)
print(torch.allclose(softmax(x3), torch.softmax(x3, dim=0)))  # True

tensor([0.09003057, 0.24472848, 0.66524094])
tensor(1.)
True
True


In [8]:
def layer_norm(x, g_b, eps: float = 1e-5):
    """
        1.计算每一个样本的均值mean和方差var
        2.对输入的张量进行标准化
            T(b,s,c)={[T(b,s,c)-mean]/(var+eps)^(1/2)}*gamma+bias
        3.对上一步的结果进行缩放和加偏置
    """

    #从g_b字典中取出缩放系数g(gamma)和偏置量b(bias)
    g, b = torch.Tensor(g_b['g']), torch.Tensor(g_b['b'])

    #计算均值和方差
    #unbiased=False表示计算方差时除以n，而不是n-1，这样得到的是“总体方差”
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)

    #标准化
    x_norm = (x - mean) / torch.sqrt(var + eps)

    #缩放、偏置
    result = x_norm * g + b

    return result

In [9]:
import torch.nn as nn


# 你的实现
def layer_norm(x, g_b, eps: float = 1e-5):
    g, b = torch.Tensor(g_b['g']), torch.Tensor(g_b['b'])
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, keepdim=True, unbiased=False)
    x_norm = (x - mean) / torch.sqrt(var + eps)
    return x_norm * g + b


# 测试数据
x_test = torch.randn(2, 4, 8)
g_b = {'g': torch.ones(8), 'b': torch.zeros(8)}  # 初始状态：gamma=1, beta=0

# 对比官方实现
official_ln = nn.LayerNorm(8, eps=1e-5)
official_ln.weight.data = torch.ones(8)  # 重置为初始状态
official_ln.bias.data = torch.zeros(8)

my_result = layer_norm(x_test, g_b)
official_result = official_ln(x_test)

print(torch.allclose(my_result, official_result, rtol=1e-4, atol=1e-5))

True


In [10]:
def linear(x, w_b):  # [m, in], [in, out], [out] -> [m, out]
    """
        y = x @ w + b
    """
    w, b = w_b['w'], w_b['b']
    y = x @ w + b
    return y

In [13]:
# 测试
x = torch.randn(2, 4, 768)
w_b = {'w': torch.randn(768, 2304), 'b': torch.randn(2304)}
y = linear(x, w_b)
print(y.shape)  # torch.Size([2, 4, 2304]) ✅

torch.Size([2, 4, 2304])


In [14]:
def ffn(x, mlp):  # [n_seq, n_embd] -> [n_seq, n_embd]
    """
        Feed-Forward Network，前馈神经网络
        FFN(x) = GELU(x @ w1 + b1) @ w2 + b2
    """
    w_b1, w_b2 = mlp['c_fc'], mlp['c_proj']
    y = linear(gelu(linear(x, w_b1)), w_b2)
    return y

In [1]:
def attention(q, k, v, mask):  # [n_q, d_k], [n_k, d_k], [n_k, d_v], [n_q, n_k] -> [n_q, d_v]
    """
        mha:
            Q = q @ I
            K = k @ I
            V = v @ I

        attention:
        1.计算相似度矩阵
            A = Q @ K^T
        2.缩放点积注意力
            scores/=d_k^(1/2)
        3.加掩码
        4.softmax归一化得到A'
        5.加权求和
            O = A' @ V
    """
    #1
    A = q @ k.transpose(-2, -1)
    #2
    d_k = q.size(-1)
    A = A / math.sqrt(d_k)
    #3 将掩码矩阵为-inf的位置赋值为一个非常大的负数
    A = A.masked_fill(mask, -1e9)
    #4
    A_ = softmax(A, dim=-1)
    #5
    O = A_ @ v

    return O

In [2]:
def mha(x, attn, n_head):  # [n_seq, n_embd] -> [n_seq, n_embd]

    c_attn, c_proj = attn['c_attn'], attn['c_proj']
    # qkv projection
    x = linear(x, c_attn)  # [n_seq, n_embd] -> [n_seq, 3*n_embd]

    #拆分qkv
    qkv = torch.chunk(x, 3, dim=-1)

    # Split into heads
    qkv_heads = [qkv_part.chunk(n_head, dim=-1) for qkv_part in
                 qkv]  # 3 * [n_seq, n_embd] -> 3 * n_head * [n_seq, n_embd/n_head]
    qkv_heads = list(zip(*qkv_heads))  # [3, n_head, n_seq, n_embd/n_head]

    #构造上三角矩阵causal_mask
    """
            | 0  -inf -inf ... -inf |
            | 0    0  -inf ... -inf |
            | 0    0    0  ... -inf |
            |...  ...  ... ...  ... |
            | 0    0    0  ...   0  |
    """
    n_seq = x.size(0)
    #生成一个主对角线及以下为False，以上为True的三角矩阵
    causal_mask = torch.triu(torch.ones(n_seq, n_seq), diagonal=1).bool()

    # Perform attention over each head
    out_heads = [attention(q, k, v, causal_mask) for q, k, v in qkv_heads]  # n_head * [n_seq, n_embd/n_head]

    #合并多头
    x = torch.cat(out_heads, dim=-1)

    # Out projection
    x = linear(x, c_proj)  # [n_seq, n_embd] -> [n_seq, n_embd]

    return x